In [3]:
# extractor.py
# End-to-end extractor with:
# 1) FIBA-style event normalization (P2, P3, FT, REB, TREB, ASS, TO, ST, BS, FOUL, RFOUL)
# 2) Possession tagging (possession_id, owner, end reasons)
# 3) Team + possession-level plus-minus
# 4) Lineup stint inference from substitutions (graceful fallback if none)
# 5) Robust position extraction + normalization (Guard/Forward/Center)
#
# Inputs:
#   - data.json (Genius Sports format)
#   - [optional] positions_override.csv with any of:
#       player_no, position
#       team_code, shirtNumber, position
#       name, position
#     'position' can be PG, G, SG, SF, F, PF, C, G/F, F/C, Guard, etc.
#
# Outputs (in current folder):
#   - game_pbp_normalized.csv
#   - game_possessions.csv
#   - game_team_pm.csv
#   - game_stints.csv
#   - game_stints_pm.csv
#   - game_players_with_positions.csv
#   - game_player_positions_lookup.csv
#
# Notes:
# - Defensive team rebounds (TREB) end possessions by design.
# - Position normalization maps:
#       PG, SG, G -> Guard
#       SF, PF, F -> Forward
#       C        -> Center
#       Hybrids (e.g., G/F, F-C) -> Guard-Forward, Forward-Center

import json
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import pandas as pd

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
SRC = Path("data.json")  # change to Path("/mnt/data/data.json") if needed
OVERRIDE = Path("positions_override.csv")  # optional

# ---------------------------------------------------------------------
# Utilities: positions
# ---------------------------------------------------------------------
_POS_SYNONYMS = {
    "pg": "Guard", "point": "Guard", "pointguard": "Guard", "guard": "Guard", "g": "Guard", "sg": "Guard",
    "sf": "Forward", "pf": "Forward", "forward": "Forward", "f": "Forward",
    "c": "Center", "center": "Center",
}

def _clean_str(x: Optional[str]) -> str:
    return str(x).strip() if x is not None else ""

def normalize_position_label(pos: Optional[str]) -> Tuple[str, str]:
    """
    Returns (pos_raw, pos_group) where:
      - pos_raw: the best raw label we found from the roster/overrides (original-ish)
      - pos_group: Guard / Forward / Center / Guard-Forward / Forward-Center / "" if unknown
    """
    raw = _clean_str(pos)
    if not raw:
        return "", ""

    # normalize separators and lowercase
    s = raw.replace(" ", "").replace("\\", "/").replace("-", "/").lower()

    # handle hybrids like G/F, F/C, GF, FC
    if s in ("gf", "g/f", "fg", "f/g"):
        return raw, "Guard-Forward"
    if s in ("fc", "f/c", "cf", "c/f"):
        return raw, "Forward-Center"

    # map single tokens or words
    # try a few common full words first
    if s in _POS_SYNONYMS:
        return raw, _POS_SYNONYMS[s]

    # further heuristics
    # e.g., "shootingguard", "smallforward", "powerforward"
    if "guard" in s:
        return raw, "Guard"
    if "forward" in s:
        return raw, "Forward"
    if "center" in s or s == "c5" or s == "big":
        return raw, "Center"

    # two-letter combos like "sg", "pg", "pf", "sf"
    if len(s) <= 2 and s in _POS_SYNONYMS:
        return raw, _POS_SYNONYMS[s]

    # still unknown
    return raw, ""

def coalesce_position_dict(p: Dict) -> Optional[str]:
    """
    Try several plausible roster keys to find a position label.
    """
    for k in ("playingPosition", "position", "pos", "primaryPosition", "positionShort", "role"):
        v = p.get(k)
        if v is not None and str(v).strip():
            return str(v)
    return None

# ---------------------------------------------------------------------
# Core builders
# ---------------------------------------------------------------------
def build_team_map(raw) -> Dict[int, Dict]:
    tmap = {}
    for tno, t in raw.get("tm", {}).items():
        tmap[int(tno)] = {
            "team_no": int(tno),
            "team_name": t.get("name"),
            "team_code": t.get("code"),
            "shortName": t.get("shortName"),
        }
    return tmap

def build_player_map(raw) -> Dict[int, Dict]:
    pmap = {}
    for tno, t in raw.get("tm", {}).items():
        for pid, p in t.get("pl", {}).items():
            pos_raw_guess = coalesce_position_dict(p)
            pos_raw, pos_group = normalize_position_label(pos_raw_guess)

            pmap[int(pid)] = {
                "player_no": int(pid),
                "team_no": int(tno),
                "firstName": p.get("firstName"),
                "familyName": p.get("familyName"),
                "name": p.get("name") or f"{p.get('firstName','')} {p.get('familyName','')}".strip(),
                "shirtNumber": p.get("shirtNumber"),
                "playingPosition_raw": pos_raw,
                "position_group": pos_group,
                "starter": p.get("starter"),
                "captain": p.get("captain"),
                "active": p.get("active"),
            }
    return pmap

def normalize_row(ev: pd.Series) -> Tuple[str, Optional[str], str]:
    """
    Map Genius actionType/subType to FIBA-like:
    P2, P3, FT, REB, TREB, ASS, TO, ST, BS, FOUL, RFOUL, TIMEOUT, SUB, JUMP, VIOL, OTHER
    result: 'made'/'missed' for shots/FT; else None
    """
    at = _clean_str(ev.get("actionType")).lower()
    st = _clean_str(ev.get("subType")).lower()
    success = ev.get("success")
    player = ev.get("player") or ev.get("player_name_from_roster") or ""
    team_name = ev.get("team_name") or ""

    if at in ("2pt","2pointer","2points"):
        return ("P2", "made" if success == 1 else "missed", f"{team_name} {player} 2Pt {'made' if success==1 else 'missed'}")
    if at in ("3pt","3pointer","3points"):
        return ("P3", "made" if success == 1 else "missed", f"{team_name} {player} 3Pt {'made' if success==1 else 'missed'}")
    if at in ("freethrow","free-throw","ft"):
        return ("FT", "made" if success == 1 else "missed", f"{team_name} {player} FT {st or ''} {'made' if success==1 else 'missed'}".strip())
    if at == "rebound":
        is_team = (not ev.get("player")) and (not ev.get("shirtNumber"))
        code = "TREB" if is_team else "REB"
        side = st or ""
        return (code, None, f"{team_name} {'team ' if is_team else ''}rebound {side}".strip())
    if at == "assist":
        return ("ASS", None, f"{team_name} {player} assist")
    if at == "turnover":
        return ("TO", None, f"{team_name} {player} turnover {st}".strip())
    if at == "steal":
        return ("ST", None, f"{team_name} {player} steal")
    if at == "block":
        return ("BS", None, f"{team_name} {player} block")
    if at == "foul":
        code = "RFOUL" if "offensive" in st else "FOUL"
        return (code, None, f"{team_name} {player} foul {st}".strip())
    if at == "timeout":
        return ("TIMEOUT", None, f"{team_name} timeout")
    if at in ("sub","substitution"):
        return ("SUB", None, f"{team_name} substitution")
    if at in ("jumpball","jump"):
        return ("JUMP", None, "jump ball")
    if at in ("violation","travel","double-dribble","lane-violation","lane"):
        return ("VIOL", None, f"{team_name} violation {st}".strip())
    return ("OTHER", None, f"{team_name} {at} {st}".strip())

def event_points(ev: pd.Series) -> int:
    if ev["ev_code"] == "P2" and ev["ev_result"] == "made":
        return 2
    if ev["ev_code"] == "P3" and ev["ev_result"] == "made":
        return 3
    if ev["ev_code"] == "FT" and ev["ev_result"] == "made":
        return 1
    return 0

def possession_change(ev_curr: pd.Series, ev_next: Optional[pd.Series]) -> Tuple[bool, Optional[str]]:
    """
    Heuristics:
    - Made FG or made last FT (1of1/2of2/3of3) -> change
    - Turnover or offensive foul -> change
    - Defensive rebound (incl. team TREB) after a miss -> change
    - If current event itself is defensive rebound/TREB -> change
    """
    code = ev_curr["ev_code"]
    st = _clean_str(ev_curr.get("subType")).lower()

    if code in ("P2","P3","FT") and ev_curr["ev_result"] == "made":
        if code in ("P2","P3"):
            return True, "score"
        subtype = st.replace(" ", "")
        if any(tag in subtype for tag in ["1of1","2of2","3of3"]):
            return True, "made_ft_last"
        return True, "made_ft"

    if code == "TO":
        return True, "turnover"
    if code == "RFOUL":
        return True, "offensive_foul"

    if code in ("P2","P3","FT") and ev_curr["ev_result"] == "missed":
        if ev_next is not None and ev_next["ev_code"] in ("REB","TREB"):
            next_st = _clean_str(ev_next.get("subType")).lower()
            if "defensive" in next_st or ev_next["ev_code"] == "TREB":
                return True, "def_reb"

    if code in ("REB","TREB"):
        if "defensive" in st or code == "TREB":
            return True, "def_reb"

    return False, None

def summarize_possessions(df: pd.DataFrame, team_map: Dict[int, Dict]) -> pd.DataFrame:
    g = df.groupby("possession_id", as_index=False).agg(
        period=("period","first"),
        start_action=("actionNumber","first"),
        end_action=("actionNumber","last"),
        start_clock=("clock","first"),
        end_clock=("clock","last"),
        owner_tno=("possession_owner_tno","first"),
        end_reason=("possession_end_reason","last"),
        pts_owner=("points", "sum"),
    )
    # pts_against: sum of points where tno != owner
    g["pts_against"] = g["possession_id"].map(
        lambda pid: int(df[(df["possession_id"] == pid) & (df["tno"] != g.loc[g["possession_id"] == pid, "owner_tno"].iloc[0])]["points"].sum())
    )
    g["owner_team_name"] = g["owner_tno"].map(lambda t: team_map.get(int(t), {}).get("team_name") if pd.notna(t) else None)
    g["net_pts"] = g["pts_owner"] - g["pts_against"]
    return g

def initial_lineup_for_team(team_no: int, pbp_df: pd.DataFrame, player_map: Dict[int, Dict]) -> List[int]:
    # 1) Prefer explicit starters
    starters = [pid for pid, info in player_map.items() if info["team_no"]==team_no and (info.get("starter")==1 or info.get("starter")==True)]
    if len(starters) >= 5:
        return starters[:5]
    # 2) Otherwise, first five distinct players appearing in P1
    seen = []
    for _, ev in pbp_df[pbp_df["period"]==1].iterrows():
        if ev.get("tno")==team_no and pd.notna(ev.get("pno")):
            pid = int(ev["pno"])
            if pid not in seen:
                seen.append(pid)
            if len(seen)==5:
                break
    # 3) If still <5, pad by most minutes on the roster (common data gap)
    if len(seen) < 5:
        # minutes often keyed as sMinutes "mm:ss"; fallback to active flag
        # we'll just pick any remaining teammates to reach 5
        rest = [pid for pid, info in player_map.items() if info["team_no"]==team_no and pid not in seen]
        seen.extend(rest[: max(0, 5-len(seen)) ])
    return seen[:5]

def infer_stints(pbp_df: pd.DataFrame, team_map: Dict[int, Dict], player_map: Dict[int, Dict]) -> pd.DataFrame:
    subs = pbp_df[pbp_df["ev_code"]=="SUB"]
    if subs.empty:
        rows = []
        for team_no in sorted(team_map.keys()):
            lineup = initial_lineup_for_team(team_no, pbp_df, player_map)
            rows.append({
                "team_no": team_no,
                "team_name": team_map[team_no]["team_name"],
                "start_action": pbp_df["actionNumber"].min(),
                "end_action": pbp_df["actionNumber"].max(),
                "players_on_court": lineup,
            })
        return pd.DataFrame(rows)

    rows = []
    for team_no in sorted(team_map.keys()):
        current = set(initial_lineup_for_team(team_no, pbp_df, player_map))
        start_action = pbp_df["actionNumber"].min()
        for _, ev in pbp_df[pbp_df["tno"]==team_no].iterrows():
            if ev["ev_code"]!="SUB":
                continue
            in_pid = int(ev["pno"]) if pd.notna(ev.get("pno")) else None
            end_action = ev["actionNumber"]
            rows.append({
                "team_no": team_no,
                "team_name": team_map[team_no]["team_name"],
                "start_action": start_action,
                "end_action": end_action,
                "players_on_court": sorted(list(current)),
            })
            if in_pid is not None:
                if len(current) >= 5:
                    # if 'out' not logged, drop an arbitrary player (least informative but keeps cardinality)
                    current.pop()
                current.add(in_pid)
            start_action = end_action + 0.1
        rows.append({
            "team_no": team_no,
            "team_name": team_map[team_no]["team_name"],
            "start_action": start_action,
            "end_action": pbp_df["actionNumber"].max(),
            "players_on_court": sorted(list(current)),
        })
    return pd.DataFrame(rows)

def stint_plus_minus(stints: pd.DataFrame, poss: pd.DataFrame) -> pd.DataFrame:
    if stints.empty or poss.empty:
        return pd.DataFrame()
    rows = []
    for _, st in stints.iterrows():
        mask = (
            (poss["owner_tno"]==st["team_no"]) &
            (poss["start_action"]>=st["start_action"]) &
            (poss["end_action"]<=st["end_action"])
        )
        seg = poss[mask]
        rows.append({
            "team_no": st["team_no"],
            "team_name": st["team_name"],
            "start_action": st["start_action"],
            "end_action": st["end_action"],
            "players_on_court": st["players_on_court"],
            "possessions": int(seg["possession_id"].nunique()),
            "pts_for": int(seg["pts_owner"].sum()),
            "pts_against": int(seg["pts_against"].sum()),
            "net": int(seg["net_pts"].sum()),
        })
    return pd.DataFrame(rows)

def lineup_pos_mix(players_on_court: List[int], player_map: Dict[int, Dict]) -> str:
    groups = [player_map.get(pid, {}).get("position_group","") for pid in players_on_court]
    g = sum(1 for x in groups if x == "Guard")
    f = sum(1 for x in groups if x == "Forward")
    c = sum(1 for x in groups if x == "Center")
    # treat hybrid as both halves (optional); here we just show as Unknown if empty
    u = sum(1 for x in groups if not x or x not in ("Guard","Forward","Center"))
    parts = []
    if g: parts.append(f"Gx{g}")
    if f: parts.append(f"Fx{f}")
    if c: parts.append(f"Cx{c}")
    if u: parts.append(f"Unknownx{u}")
    return ",".join(parts)

# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main():
    assert SRC.exists(), f"Missing {SRC}"
    raw = json.loads(SRC.read_text(encoding="utf-8"))

    team_map = build_team_map(raw)
    player_map = build_player_map(raw)

    # play-by-play
    pbp_df = pd.DataFrame(raw["pbp"])
    if "actionNumber" in pbp_df.columns:
        pbp_df["actionNumber"] = pd.to_numeric(pbp_df["actionNumber"], errors="coerce")
    pbp_df = pbp_df.sort_values(["period","actionNumber"]).reset_index(drop=True)

    for c in ["tno","pno","period","lead"]:
        if c in pbp_df.columns:
            pbp_df[c] = pd.to_numeric(pbp_df[c], errors="coerce")

    pbp_df["team_name"] = pbp_df["tno"].map(lambda t: team_map.get(int(t), {}).get("team_name") if pd.notna(t) else None)
    pbp_df["team_code"] = pbp_df["tno"].map(lambda t: team_map.get(int(t), {}).get("team_code") if pd.notna(t) else None)
    pbp_df["player_name_from_roster"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("name") if pd.notna(p) else None)
    pbp_df["player_shirt"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("shirtNumber") if pd.notna(p) else None)

    # Positions into PBP (raw + group)
    pbp_df["player_pos_raw"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("playingPosition_raw") if pd.notna(p) else None)
    pbp_df["player_pos_group"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)

    # shots table -> merge x,y,r
    shot_rows = []
    for tno, t in raw.get("tm", {}).items():
        for s in t.get("shot", []) or []:
            rec = s.copy()
            rec["tno"] = int(tno)
            shot_rows.append(rec)
    shots_df = pd.DataFrame(shot_rows) if shot_rows else pd.DataFrame()
    if not shots_df.empty:
        for c in ["x","y","r","pno","tno","actionNumber","per"]:
            if c in shots_df.columns:
                shots_df[c] = pd.to_numeric(shots_df[c], errors="coerce")
        shots_df.rename(columns={"per":"period"}, inplace=True)
        pbp_df = pbp_df.merge(shots_df[["actionNumber","x","y","r"]], on="actionNumber", how="left")

    # normalize to FIBA-like
    norm = pbp_df.apply(normalize_row, axis=1, result_type="expand")
    pbp_df[["ev_code","ev_result","ev_text"]] = norm

    # assist/block stitching (and attach their positions)
    assist_map, block_map = {}, {}
    for _, ev in pbp_df.iterrows():
        if ev.get("actionType") == "assist" and pd.notna(ev.get("previousAction")):
            assist_map[ev["previousAction"]] = {
                "assist_pno": ev.get("pno"),
                "assist_player": ev.get("player") or ev.get("player_name_from_roster"),
            }
        if ev.get("actionType") == "block" and pd.notna(ev.get("previousAction")):
            block_map[ev["previousAction"]] = {
                "block_pno": ev.get("pno"),
                "block_player": ev.get("player") or ev.get("player_name_from_roster"),
            }

    is_shot = pbp_df["actionType"].isin(["2pt","3pt"])
    pbp_df.loc[is_shot, "assist_pno"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: assist_map.get(an, {}).get("assist_pno"))
    pbp_df.loc[is_shot, "assist_player"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: assist_map.get(an, {}).get("assist_player"))
    pbp_df.loc[is_shot, "block_pno"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: block_map.get(an, {}).get("block_pno"))
    pbp_df.loc[is_shot, "block_player"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: block_map.get(an, {}).get("block_player"))

    # positions for assist/block actors
    pbp_df["assist_pos_group"] = pbp_df["assist_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)
    pbp_df["block_pos_group"]  = pbp_df["block_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group")  if pd.notna(p) else None)

    # points + possessions
    pbp_df["points"] = pbp_df.apply(event_points, axis=1)

    pos_id = 0
    curr_pos_team: Optional[int] = None
    pos_ids: List[int] = []
    pos_owners: List[Optional[int]] = []
    pos_end_reason: List[Optional[str]] = []

    rows = pbp_df.to_dict(orient="records")
    for i, ev in enumerate(rows):
        if curr_pos_team is None:
            if ev["ev_code"] in ("P2","P3","FT","TO","RFOUL"):
                curr_pos_team = ev.get("tno")
                pos_id += 1

        pos_ids.append(pos_id if curr_pos_team is not None else 0)
        pos_owners.append(curr_pos_team)

        nxt = rows[i+1] if i+1 < len(rows) else None
        is_change, reason = possession_change(ev, nxt)
        if is_change:
            pos_end_reason.append(reason)
            if curr_pos_team in (1,2):  # quick toggle for 2-team feeds
                curr_pos_team = 3 - curr_pos_team
            else:
                curr_pos_team = nxt.get("tno") if nxt is not None else None
            pos_id += 1
        else:
            pos_end_reason.append(None)

    pbp_df["possession_id"] = pos_ids
    pbp_df["possession_owner_tno"] = pos_owners
    pbp_df["possession_end_reason"] = pos_end_reason

    possum_df = summarize_possessions(pbp_df, team_map)
    team_pm = possum_df.groupby("owner_team_name", as_index=False).agg(
        poss=("possession_id","nunique"),
        pts_for=("pts_owner","sum"),
        pts_against=("pts_against","sum"),
        net=("net_pts","sum")
    )

    stints_df = infer_stints(pbp_df, team_map, player_map)
    stints_pm_df = stint_plus_minus(stints_df, possum_df)

    # Attach lineup position mix to stints
    if not stints_df.empty:
        stints_df["lineup_pos_mix"] = stints_df["players_on_court"].map(lambda lst: lineup_pos_mix(lst, player_map))
        stints_df["lineup_size"] = stints_df["players_on_court"].map(lambda lst: len(lst))

    # players + positions table (wide, includes s* stat keys and pos normalization)
    players_rows = []
    for tno, t in raw.get("tm", {}).items():
        for pid, p in t.get("pl", {}).items():
            row = {
                "team_no": int(tno),
                "team_name": t.get("name"),
                "team_code": t.get("code"),
                "player_no": int(pid),
            }
            for k in ["name","firstName","familyName","shirtNumber","starter","captain","active"]:
                row[k] = p.get(k)

            # best position guess + normalized group
            pos_guess = coalesce_position_dict(p)
            pos_raw, pos_group = normalize_position_label(pos_guess)
            row["position_raw"] = pos_raw
            row["position_group"] = pos_group

            # include any s*-prefixed stats on the roster node
            for k, v in p.items():
                if k.startswith("s"):
                    row[k] = v
            players_rows.append(row)

    players_df = pd.DataFrame(players_rows)

    # Optional: apply overrides if provided
    if OVERRIDE.exists():
        ov = pd.read_csv(OVERRIDE)
        ov.columns = [c.strip() for c in ov.columns]
        ov["position"] = ov["position"].astype(str)

        # normalize incoming override position
        ov["position_raw"], ov["position_group"] = zip(*ov["position"].map(normalize_position_label))

        # join priority: player_no -> (team_code, shirtNumber) -> name
        # 1) player_no
        if "player_no" in ov.columns:
            players_df = players_df.merge(
                ov[["player_no","position_raw","position_group"]],
                on="player_no", how="left", suffixes=("","_ov1")
            )
            players_df["position_raw"]   = players_df["position_raw_ov1"].combine_first(players_df["position_raw"])
            players_df["position_group"] = players_df["position_group_ov1"].combine_first(players_df["position_group"])
            players_df.drop(columns=[c for c in players_df.columns if c.endswith("_ov1")], inplace=True)

        # 2) (team_code, shirtNumber)
        if {"team_code","shirtNumber"}.issubset(ov.columns):
            players_df = players_df.merge(
                ov[["team_code","shirtNumber","position_raw","position_group"]],
                on=["team_code","shirtNumber"], how="left", suffixes=("","_ov2")
            )
            players_df["position_raw"]   = players_df["position_raw_ov2"].combine_first(players_df["position_raw"])
            players_df["position_group"] = players_df["position_group_ov2"].combine_first(players_df["position_group"])
            players_df.drop(columns=[c for c in players_df.columns if c.endswith("_ov2")], inplace=True)

        # 3) name
        if "name" in ov.columns:
            players_df = players_df.merge(
                ov[["name","position_raw","position_group"]],
                on="name", how="left", suffixes=("","_ov3")
            )
            players_df["position_raw"]   = players_df["position_raw_ov3"].combine_first(players_df["position_raw"])
            players_df["position_group"] = players_df["position_group_ov3"].combine_first(players_df["position_group"])
            players_df.drop(columns=[c for c in players_df.columns if c.endswith("_ov3")], inplace=True)

        # reflect final positions back into player_map for downstream usage
        for _, r in players_df[["player_no","position_raw","position_group"]].iterrows():
            if int(r["player_no"]) in player_map:
                player_map[int(r["player_no"])]["playingPosition_raw"] = r["position_raw"]
                player_map[int(r["player_no"])]["position_group"] = r["position_group"]

        # refresh pbp position columns (assist/block may change)
        pbp_df["player_pos_raw"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("playingPosition_raw") if pd.notna(p) else None)
        pbp_df["player_pos_group"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)
        pbp_df["assist_pos_group"] = pbp_df["assist_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)
        pbp_df["block_pos_group"]  = pbp_df["block_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group")  if pd.notna(p) else None)

        if not stints_df.empty:
            stints_df["lineup_pos_mix"] = stints_df["players_on_court"].map(lambda lst: lineup_pos_mix(lst, player_map))

    # a small positions-only lookup
    pos_lookup = players_df[[
        "team_no","team_name","team_code","player_no","name","shirtNumber","position_raw","position_group"
    ]].sort_values(["team_no","name"])

    # -----------------------------------------------------------------
    # WRITE
    # -----------------------------------------------------------------
    OUT = Path(".")
    OUT.mkdir(parents=True, exist_ok=True)
    pbp_df.to_csv(OUT/"game_pbp_normalized.csv", index=False)
    possum_df.to_csv(OUT/"game_possessions.csv", index=False)
    team_pm.to_csv(OUT/"game_team_pm.csv", index=False)
    stints_df.to_csv(OUT/"game_stints.csv", index=False)
    stints_pm_df.to_csv(OUT/"game_stints_pm.csv", index=False)
    players_df.to_csv(OUT/"game_players_with_positions.csv", index=False)
    pos_lookup.to_csv(OUT/"game_player_positions_lookup.csv", index=False)

    # Quick console summary
    total_players = len(players_df)
    known_groups = players_df["position_group"].notna() & (players_df["position_group"]!="")
    print(f"[positions] {known_groups.sum()}/{total_players} players have a normalized position group.")
    if OVERRIDE.exists():
        print(f"[positions] overrides applied from {OVERRIDE}")

if __name__ == "__main__":
    main()


[positions] 0/24 players have a normalized position group.


In [5]:
# extractor.py
# End-to-end extractor with:
# 1) FIBA-style event normalization (P2, P3, FT, REB, TREB, ASS, TO, ST, BS, FOUL, RFOUL)
# 2) Possession tagging (possession_id, owner, end reasons)
# 3) Team + possession-level plus-minus
# 4) Lineup stint inference from substitutions (graceful fallback if none)
# 5) Robust position extraction + normalization (Guard/Forward/Center)
# 6) Stint timing + scoreboard snapshots; lineup-vs-lineup matchup segments with ORtg/DRtg
#
# Inputs:
#   - data.json (Genius Sports format)
#   - [optional] positions_override.csv with any of:
#       player_no, position
#       team_code, shirtNumber, position
#       name, position
#     'position' can be PG, G, SG, SF, F, PF, C, G/F, F/C, Guard, etc.
#
# Outputs (in current folder):
#   - game_pbp_normalized.csv
#   - game_possessions.csv
#   - game_team_pm.csv
#   - game_stints.csv
#   - game_stints_pm.csv
#   - game_lineup_matchups.csv
#   - game_players_with_positions.csv
#   - game_player_positions_lookup.csv

import json
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import pandas as pd

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
SRC = Path("data.json")               # change to Path("/mnt/data/data.json") if needed
OVERRIDE = Path("positions_override.csv")  # optional

# ---------------------------------------------------------------------
# Utilities: positions
# ---------------------------------------------------------------------
_POS_SYNONYMS = {
    "pg": "Guard", "point": "Guard", "pointguard": "Guard", "guard": "Guard", "g": "Guard", "sg": "Guard",
    "sf": "Forward", "pf": "Forward", "forward": "Forward", "f": "Forward",
    "c": "Center", "center": "Center",
}

def _clean_str(x: Optional[str]) -> str:
    return str(x).strip() if x is not None else ""

def normalize_position_label(pos: Optional[str]) -> Tuple[str, str]:
    """
    Returns (pos_raw, pos_group) where pos_group in:
    Guard / Forward / Center / Guard-Forward / Forward-Center / "" (unknown)
    """
    raw = _clean_str(pos)
    if not raw:
        return "", ""

    s = raw.replace(" ", "").replace("\\", "/").replace("-", "/").lower()

    if s in ("gf", "g/f", "fg", "f/g"):
        return raw, "Guard-Forward"
    if s in ("fc", "f/c", "cf", "c/f"):
        return raw, "Forward-Center"

    if s in _POS_SYNONYMS:
        return raw, _POS_SYNONYMS[s]

    if "guard" in s:
        return raw, "Guard"
    if "forward" in s:
        return raw, "Forward"
    if "center" in s or s == "c5" or s == "big":
        return raw, "Center"

    if len(s) <= 2 and s in _POS_SYNONYMS:
        return raw, _POS_SYNONYMS[s]

    return raw, ""

def coalesce_position_dict(p: Dict) -> Optional[str]:
    for k in ("playingPosition", "position", "pos", "primaryPosition", "positionShort", "role"):
        v = p.get(k)
        if v is not None and str(v).strip():
            return str(v)
    return None

# ---------------------------------------------------------------------
# Core builders
# ---------------------------------------------------------------------
def build_team_map(raw) -> Dict[int, Dict]:
    tmap = {}
    for tno, t in raw.get("tm", {}).items():
        tmap[int(tno)] = {
            "team_no": int(tno),
            "team_name": t.get("name"),
            "team_code": t.get("code"),
            "shortName": t.get("shortName"),
        }
    return tmap

def build_player_map(raw) -> Dict[int, Dict]:
    pmap = {}
    for tno, t in raw.get("tm", {}).items():
        for pid, p in t.get("pl", {}).items():
            pos_raw_guess = coalesce_position_dict(p)
            pos_raw, pos_group = normalize_position_label(pos_raw_guess)
            pmap[int(pid)] = {
                "player_no": int(pid),
                "team_no": int(tno),
                "firstName": p.get("firstName"),
                "familyName": p.get("familyName"),
                "name": p.get("name") or f"{p.get('firstName','')} {p.get('familyName','')}".strip(),
                "shirtNumber": p.get("shirtNumber"),
                "playingPosition_raw": pos_raw,
                "position_group": pos_group,
                "starter": p.get("starter"),
                "captain": p.get("captain"),
                "active": p.get("active"),
            }
    return pmap

def normalize_row(ev: pd.Series) -> Tuple[str, Optional[str], str]:
    at = _clean_str(ev.get("actionType")).lower()
    st = _clean_str(ev.get("subType")).lower()
    success = ev.get("success")
    player = ev.get("player") or ev.get("player_name_from_roster") or ""
    team_name = ev.get("team_name") or ""

    if at in ("2pt","2pointer","2points"):
        return ("P2", "made" if success == 1 else "missed", f"{team_name} {player} 2Pt {'made' if success==1 else 'missed'}")
    if at in ("3pt","3pointer","3points"):
        return ("P3", "made" if success == 1 else "missed", f"{team_name} {player} 3Pt {'made' if success==1 else 'missed'}")
    if at in ("freethrow","free-throw","ft"):
        return ("FT", "made" if success == 1 else "missed", f"{team_name} {player} FT {st or ''} {'made' if success==1 else 'missed'}".strip())
    if at == "rebound":
        is_team = (not ev.get("player")) and (not ev.get("shirtNumber"))
        code = "TREB" if is_team else "REB"
        side = st or ""
        return (code, None, f"{team_name} {'team ' if is_team else ''}rebound {side}".strip())
    if at == "assist":
        return ("ASS", None, f"{team_name} {player} assist")
    if at == "turnover":
        return ("TO", None, f"{team_name} {player} turnover {st}".strip())
    if at == "steal":
        return ("ST", None, f"{team_name} {player} steal")
    if at == "block":
        return ("BS", None, f"{team_name} {player} block")
    if at == "foul":
        code = "RFOUL" if "offensive" in st else "FOUL"
        return (code, None, f"{team_name} {player} foul {st}".strip())
    if at == "timeout":
        return ("TIMEOUT", None, f"{team_name} timeout")
    if at in ("sub","substitution"):
        return ("SUB", None, f"{team_name} substitution")
    if at in ("jumpball","jump"):
        return ("JUMP", None, "jump ball")
    if at in ("violation","travel","double-dribble","lane-violation","lane"):
        return ("VIOL", None, f"{team_name} violation {st}".strip())
    return ("OTHER", None, f"{team_name} {at} {st}".strip())

def event_points(ev: pd.Series) -> int:
    if ev["ev_code"] == "P2" and ev["ev_result"] == "made":
        return 2
    if ev["ev_code"] == "P3" and ev["ev_result"] == "made":
        return 3
    if ev["ev_code"] == "FT" and ev["ev_result"] == "made":
        return 1
    return 0

def possession_change(ev_curr: pd.Series, ev_next: Optional[pd.Series]) -> Tuple[bool, Optional[str]]:
    code = ev_curr["ev_code"]
    st = _clean_str(ev_curr.get("subType")).lower()

    if code in ("P2","P3","FT") and ev_curr["ev_result"] == "made":
        if code in ("P2","P3"):
            return True, "score"
        subtype = st.replace(" ", "")
        if any(tag in subtype for tag in ["1of1","2of2","3of3"]):
            return True, "made_ft_last"
        return True, "made_ft"
    if code == "TO":
        return True, "turnover"
    if code == "RFOUL":
        return True, "offensive_foul"

    if code in ("P2","P3","FT") and ev_curr["ev_result"] == "missed":
        if ev_next is not None and ev_next["ev_code"] in ("REB","TREB"):
            next_st = _clean_str(ev_next.get("subType")).lower()
            if "defensive" in next_st or ev_next["ev_code"] == "TREB":
                return True, "def_reb"
    if code in ("REB","TREB"):
        if "defensive" in st or code == "TREB":
            return True, "def_reb"
    return False, None

def summarize_possessions(df: pd.DataFrame, team_map: Dict[int, Dict]) -> pd.DataFrame:
    g = df.groupby("possession_id", as_index=False).agg(
        period=("period","first"),
        start_action=("actionNumber","first"),
        end_action=("actionNumber","last"),
        start_clock=("clock","first"),
        end_clock=("clock","last"),
        owner_tno=("possession_owner_tno","first"),
        end_reason=("possession_end_reason","last"),
        pts_owner=("points", "sum"),
    )
    # pts_against: sum of points within this possession scored by the other team
    def opp_pts(pid):
        seg = df[df["possession_id"] == pid]
        owner = g.loc[g["possession_id"] == pid, "owner_tno"].iloc[0]
        return int(seg.loc[seg["tno"] != owner, "points"].sum())
    g["pts_against"] = g["possession_id"].map(opp_pts)
    g["owner_team_name"] = g["owner_tno"].map(lambda t: team_map.get(int(t), {}).get("team_name") if pd.notna(t) else None)
    g["net_pts"] = g["pts_owner"] - g["pts_against"]
    return g

def initial_lineup_for_team(team_no: int, pbp_df: pd.DataFrame, player_map: Dict[int, Dict]) -> List[int]:
    starters = [pid for pid, info in player_map.items() if info["team_no"]==team_no and (info.get("starter")==1 or info.get("starter")==True)]
    if len(starters) >= 5:
        return starters[:5]
    seen = []
    for _, ev in pbp_df[pbp_df["period"]==1].iterrows():
        if ev.get("tno")==team_no and pd.notna(ev.get("pno")):
            pid = int(ev["pno"])
            if pid not in seen:
                seen.append(pid)
            if len(seen)==5:
                break
    if len(seen) < 5:
        rest = [pid for pid, info in player_map.items() if info["team_no"]==team_no and pid not in seen]
        seen.extend(rest[: max(0, 5-len(seen)) ])
    return seen[:5]

def infer_stints(pbp_df: pd.DataFrame, team_map: Dict[int, Dict], player_map: Dict[int, Dict]) -> pd.DataFrame:
    subs = pbp_df[pbp_df["ev_code"]=="SUB"]
    if subs.empty:
        rows = []
        for team_no in sorted(team_map.keys()):
            lineup = initial_lineup_for_team(team_no, pbp_df, player_map)
            rows.append({
                "team_no": team_no,
                "team_name": team_map[team_no]["team_name"],
                "start_action": pbp_df["actionNumber"].min(),
                "end_action": pbp_df["actionNumber"].max(),
                "players_on_court": lineup,
            })
        return pd.DataFrame(rows)

    rows = []
    for team_no in sorted(team_map.keys()):
        current = set(initial_lineup_for_team(team_no, pbp_df, player_map))
        start_action = pbp_df["actionNumber"].min()
        for _, ev in pbp_df[pbp_df["tno"]==team_no].iterrows():
            if ev["ev_code"]!="SUB":
                continue
            in_pid = int(ev["pno"]) if pd.notna(ev.get("pno")) else None
            end_action = ev["actionNumber"]
            rows.append({
                "team_no": team_no,
                "team_name": team_map[team_no]["team_name"],
                "start_action": start_action,
                "end_action": end_action,
                "players_on_court": sorted(list(current)),
            })
            if in_pid is not None:
                if len(current) >= 5:
                    current.pop()   # drop arbitrary if 'out' not specified
                current.add(in_pid)
            start_action = end_action + 0.1
        rows.append({
            "team_no": team_no,
            "team_name": team_map[team_no]["team_name"],
            "start_action": start_action,
            "end_action": pbp_df["actionNumber"].max(),
            "players_on_court": sorted(list(current)),
        })
    return pd.DataFrame(rows)

def lineup_pos_mix(players_on_court: List[int], player_map: Dict[int, Dict]) -> str:
    groups = [player_map.get(pid, {}).get("position_group","") for pid in players_on_court]
    g = sum(1 for x in groups if x == "Guard")
    f = sum(1 for x in groups if x == "Forward")
    c = sum(1 for x in groups if x == "Center")
    u = sum(1 for x in groups if not x or x not in ("Guard","Forward","Center"))
    parts = []
    if g: parts.append(f"Gx{g}")
    if f: parts.append(f"Fx{f}")
    if c: parts.append(f"Cx{c}")
    if u: parts.append(f"Unknownx{u}")
    return ",".join(parts)

def stint_plus_minus_with_ratings(stints: pd.DataFrame,
                                  poss: pd.DataFrame,
                                  pbp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign points to stints by **events inside the stint window**:
      events with actionNumber in (start_action, end_action].
    Count possessions by **possession end** inside the window:
      poss.end_action in (start_action, end_action].

    This matches the scoreboard deltas used for score_start/score_end.
    """
    if stints.empty:
        return pd.DataFrame()

    rows = []
    for _, st in stints.iterrows():
        team_no = int(st["team_no"])
        s = float(st["start_action"])
        e = float(st["end_action"])

        # 1) Points from events inside (s, e]
        evmask = (pbp_df["actionNumber"] > s) & (pbp_df["actionNumber"] <= e)
        evseg = pbp_df[evmask]
        pts_for = int(evseg.loc[evseg["tno"] == team_no, "points"].sum())
        pts_against = int(evseg.loc[(evseg["tno"].notna()) & (evseg["tno"] != team_no), "points"].sum())

        # 2) Possessions counted by their end inside (s, e]
        pmask = (poss["end_action"] > s) & (poss["end_action"] <= e)
        pseg = poss[pmask]
        poss_for = int(pseg.loc[pseg["owner_tno"] == team_no, "possession_id"].nunique())
        poss_against = int(pseg.loc[pseg["owner_tno"] != team_no, "possession_id"].nunique())

        # 3) Ratings
        ORtg = (100.0 * pts_for / poss_for) if poss_for > 0 else None
        DRtg = (100.0 * pts_against / poss_against) if poss_against > 0 else None

        rows.append({
            "team_no": st["team_no"],
            "team_name": st["team_name"],
            "start_action": st["start_action"],
            "end_action": st["end_action"],
            "players_on_court": st["players_on_court"],
            "possessions": poss_for,      # kept for compatibility
            "poss_for": poss_for,
            "poss_against": poss_against,
            "pts_for": pts_for,
            "pts_against": pts_against,
            "net": pts_for - pts_against,
            "ORtg": ORtg,
            "DRtg": DRtg,
        })

    return pd.DataFrame(rows)


# -------- Timing & Score helpers --------
def add_cumulative_scores(pbp_df: pd.DataFrame, team_nos: List[int]) -> pd.DataFrame:
    for tno in team_nos:
        pbp_df[f"pts_{tno}"] = pbp_df.apply(lambda r: int(r["points"]) if pd.notna(r.get("tno")) and int(r["tno"])==tno else 0, axis=1)
        pbp_df[f"cum_{tno}"] = pbp_df[f"pts_{tno}"].cumsum()
    return pbp_df

def score_for_team_at_action(pbp_df: pd.DataFrame, team_nos: List[int], team_no: int, action: float, inclusive: bool) -> Tuple[int,int]:
    assert len(team_nos) == 2, "This helper assumes exactly two teams."
    opp_no = team_nos[1] if team_nos[0]==team_no else team_nos[0]
    seg = pbp_df[pbp_df["actionNumber"] <= action] if inclusive else pbp_df[pbp_df["actionNumber"] < action]
    if seg.empty:
        return 0, 0
    last = seg.iloc[-1]
    return int(last[f"cum_{team_no}"]), int(last[f"cum_{opp_no}"])

def period_clock_at_action(pbp_df: pd.DataFrame, action: float, mode: str) -> Tuple[Optional[int], Optional[str]]:
    if mode == "start":
        seg = pbp_df[pbp_df["actionNumber"] >= action]
        if seg.empty:
            return None, None
        r = seg.iloc[0]
    else:  # "end"
        seg = pbp_df[pbp_df["actionNumber"] <= action]
        if seg.empty:
            return None, None
        r = seg.iloc[-1]
    return (int(r["period"]) if pd.notna(r.get("period")) else None, r.get("clock"))

def build_lineup_matchups(stints_df: pd.DataFrame,
                          possum_df: pd.DataFrame,
                          team_nos: List[int],
                          team_map: Dict[int, Dict],
                          player_map: Dict[int, Dict],
                          pbp_df: pd.DataFrame) -> pd.DataFrame:
    """Create lineup-vs-lineup segments by intersecting team stints."""
    t1, t2 = team_nos
    A = stints_df[stints_df["team_no"]==t1].sort_values("start_action").reset_index(drop=True)
    B = stints_df[stints_df["team_no"]==t2].sort_values("start_action").reset_index(drop=True)
    i = j = 0
    rows = []
    while i < len(A) and j < len(B):
        a = A.iloc[i]
        b = B.iloc[j]
        s = max(a["start_action"], b["start_action"])
        e = min(a["end_action"],   b["end_action"])
        if s < e:
            # metrics in overlap [s, e]
            mask = (possum_df["start_action"]>=s) & (possum_df["end_action"]<=e)
            seg = possum_df[mask]

            poss_A = int(seg[seg["owner_tno"]==t1]["possession_id"].nunique())
            poss_B = int(seg[seg["owner_tno"]==t2]["possession_id"].nunique())
            pts_A_for = int(seg[seg["owner_tno"]==t1]["pts_owner"].sum())
            pts_B_for = int(seg[seg["owner_tno"]==t2]["pts_owner"].sum())
            pts_A_against = pts_B_for
            pts_B_against = pts_A_for

            ORtg_A = (100.0 * pts_A_for / poss_A) if poss_A > 0 else None
            DRtg_A = (100.0 * pts_A_against / poss_B) if poss_B > 0 else None
            ORtg_B = (100.0 * pts_B_for / poss_B) if poss_B > 0 else None
            DRtg_B = (100.0 * pts_B_against / poss_A) if poss_A > 0 else None

            sp, sc = period_clock_at_action(pbp_df, s, "start")
            ep, ec = period_clock_at_action(pbp_df, e, "end")

            team_nos_order = [t1, t2]
            sA_for, sA_against = score_for_team_at_action(pbp_df, team_nos_order, t1, s, inclusive=False)
            sB_for, sB_against = score_for_team_at_action(pbp_df, team_nos_order, t2, s, inclusive=False)
            eA_for, eA_against = score_for_team_at_action(pbp_df, team_nos_order, t1, e, inclusive=True)
            eB_for, eB_against = score_for_team_at_action(pbp_df, team_nos_order, t2, e, inclusive=True)

            rows.append({
                "start_action": s, "end_action": e,
                "start_period": sp, "start_clock": sc, "end_period": ep, "end_clock": ec,

                "teamA_no": t1, "teamA_name": team_map[t1]["team_name"],
                "lineupA_players": a["players_on_court"],
                "lineupA_pos_mix": lineup_pos_mix(a["players_on_court"], player_map),

                "teamB_no": t2, "teamB_name": team_map[t2]["team_name"],
                "lineupB_players": b["players_on_court"],
                "lineupB_pos_mix": lineup_pos_mix(b["players_on_court"], player_map),

                "poss_A": poss_A, "pts_A_for": pts_A_for, "pts_A_against": pts_A_against,
                "net_A": pts_A_for - pts_A_against, "ORtg_A": ORtg_A, "DRtg_A": DRtg_A,

                "poss_B": poss_B, "pts_B_for": pts_B_for, "pts_B_against": pts_B_against,
                "net_B": pts_B_for - pts_B_against, "ORtg_B": ORtg_B, "DRtg_B": DRtg_B,

                "score_start_A": sA_for, "score_start_B": sB_for,
                "score_end_A": eA_for, "score_end_B": eB_for,
            })

        # advance pointer whose stint ends first
        if a["end_action"] <= b["end_action"]:
            i += 1
        else:
            j += 1
    return pd.DataFrame(rows)

# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main():
    assert SRC.exists(), f"Missing {SRC}"
    raw = json.loads(SRC.read_text(encoding="utf-8"))

    team_map = build_team_map(raw)
    player_map = build_player_map(raw)

    # play-by-play
    pbp_df = pd.DataFrame(raw["pbp"])
    if "actionNumber" in pbp_df.columns:
        pbp_df["actionNumber"] = pd.to_numeric(pbp_df["actionNumber"], errors="coerce")
    pbp_df = pbp_df.sort_values(["period","actionNumber"]).reset_index(drop=True)

    for c in ["tno","pno","period","lead"]:
        if c in pbp_df.columns:
            pbp_df[c] = pd.to_numeric(pbp_df[c], errors="coerce")

    pbp_df["team_name"] = pbp_df["tno"].map(lambda t: team_map.get(int(t), {}).get("team_name") if pd.notna(t) else None)
    pbp_df["team_code"] = pbp_df["tno"].map(lambda t: team_map.get(int(t), {}).get("team_code") if pd.notna(t) else None)
    pbp_df["player_name_from_roster"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("name") if pd.notna(p) else None)
    pbp_df["player_shirt"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("shirtNumber") if pd.notna(p) else None)

    # Positions into PBP (raw + group)
    pbp_df["player_pos_raw"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("playingPosition_raw") if pd.notna(p) else None)
    pbp_df["player_pos_group"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)

    # shots table -> merge x,y,r
    shot_rows = []
    for tno, t in raw.get("tm", {}).items():
        for s in t.get("shot", []) or []:
            rec = s.copy()
            rec["tno"] = int(tno)
            shot_rows.append(rec)
    shots_df = pd.DataFrame(shot_rows) if shot_rows else pd.DataFrame()
    if not shots_df.empty:
        for c in ["x","y","r","pno","tno","actionNumber","per"]:
            if c in shots_df.columns:
                shots_df[c] = pd.to_numeric(shots_df[c], errors="coerce")
        shots_df.rename(columns={"per":"period"}, inplace=True)
        pbp_df = pbp_df.merge(shots_df[["actionNumber","x","y","r"]], on="actionNumber", how="left")

    # normalize to FIBA-like
    norm = pbp_df.apply(normalize_row, axis=1, result_type="expand")
    pbp_df[["ev_code","ev_result","ev_text"]] = norm

    # assist/block stitching (and attach their positions)
    assist_map, block_map = {}, {}
    for _, ev in pbp_df.iterrows():
        if ev.get("actionType") == "assist" and pd.notna(ev.get("previousAction")):
            assist_map[ev["previousAction"]] = {
                "assist_pno": ev.get("pno"),
                "assist_player": ev.get("player") or ev.get("player_name_from_roster"),
            }
        if ev.get("actionType") == "block" and pd.notna(ev.get("previousAction")):
            block_map[ev["previousAction"]] = {
                "block_pno": ev.get("pno"),
                "block_player": ev.get("player") or ev.get("player_name_from_roster"),
            }

    is_shot = pbp_df["actionType"].isin(["2pt","3pt"])
    pbp_df.loc[is_shot, "assist_pno"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: assist_map.get(an, {}).get("assist_pno"))
    pbp_df.loc[is_shot, "assist_player"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: assist_map.get(an, {}).get("assist_player"))
    pbp_df.loc[is_shot, "block_pno"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: block_map.get(an, {}).get("block_pno"))
    pbp_df.loc[is_shot, "block_player"] = pbp_df.loc[is_shot, "actionNumber"].map(lambda an: block_map.get(an, {}).get("block_player"))

    # positions for assist/block actors
    pbp_df["assist_pos_group"] = pbp_df["assist_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)
    pbp_df["block_pos_group"]  = pbp_df["block_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group")  if pd.notna(p) else None)

    # points + possessions
    pbp_df["points"] = pbp_df.apply(event_points, axis=1)

    pos_id = 0
    curr_pos_team: Optional[int] = None
    pos_ids: List[int] = []
    pos_owners: List[Optional[int]] = []
    pos_end_reason: List[Optional[str]] = []

    rows = pbp_df.to_dict(orient="records")
    for i, ev in enumerate(rows):
        if curr_pos_team is None:
            if ev["ev_code"] in ("P2","P3","FT","TO","RFOUL"):
                curr_pos_team = ev.get("tno")
                pos_id += 1

        pos_ids.append(pos_id if curr_pos_team is not None else 0)
        pos_owners.append(curr_pos_team)

        nxt = rows[i+1] if i+1 < len(rows) else None
        is_change, reason = possession_change(ev, nxt)
        if is_change:
            pos_end_reason.append(reason)
            if curr_pos_team in (1,2):
                curr_pos_team = 3 - curr_pos_team
            else:
                curr_pos_team = nxt.get("tno") if nxt is not None else None
            pos_id += 1
        else:
            pos_end_reason.append(None)

    pbp_df["possession_id"] = pos_ids
    pbp_df["possession_owner_tno"] = pos_owners
    pbp_df["possession_end_reason"] = pos_end_reason

    possum_df = summarize_possessions(pbp_df, team_map)
    team_pm = possum_df.groupby("owner_team_name", as_index=False).agg(
        poss=("possession_id","nunique"),
        pts_for=("pts_owner","sum"),
        pts_against=("pts_against","sum"),
        net=("net_pts","sum")
    )

    # Stints and PM with ratings
    stints_df = infer_stints(pbp_df, team_map, player_map)
    stints_df["lineup_pos_mix"] = stints_df["players_on_court"].map(lambda lst: lineup_pos_mix(lst, player_map))
    stints_df["lineup_size"] = stints_df["players_on_court"].map(lambda lst: len(lst))

    stints_pm_df = stint_plus_minus_with_ratings(stints_df, possum_df, pbp_df)

    # -------- Timing & Score snapshots for stints --------
    team_nos = sorted(list(team_map.keys()))
    pbp_df = add_cumulative_scores(pbp_df, team_nos)

    # enrich stints_df with start/end period/clock and score snapshots
    time_cols = {
        "start_period": [], "start_clock": [],
        "end_period": [],   "end_clock": [],
        "score_start_for": [], "score_start_against": [],
        "score_end_for": [],   "score_end_against": [],
    }

    for _, st in stints_df.iterrows():
        sp, sc = period_clock_at_action(pbp_df, st["start_action"], "start")
        ep, ec = period_clock_at_action(pbp_df, st["end_action"], "end")

        # score just BEFORE stint starts; and AFTER it ends (inclusive)
        s_for, s_against = score_for_team_at_action(pbp_df, team_nos, int(st["team_no"]), st["start_action"], inclusive=False)
        e_for, e_against = score_for_team_at_action(pbp_df, team_nos, int(st["team_no"]), st["end_action"], inclusive=True)

        time_cols["start_period"].append(sp)
        time_cols["start_clock"].append(sc)
        time_cols["end_period"].append(ep)
        time_cols["end_clock"].append(ec)
        time_cols["score_start_for"].append(s_for)
        time_cols["score_start_against"].append(s_against)
        time_cols["score_end_for"].append(e_for)
        time_cols["score_end_against"].append(e_against)

    for k, v in time_cols.items():
        stints_df[k] = v

    # attach same timing/score info to stints_pm_df via merge
    st_attrs = stints_df[[
        "team_no","start_action","end_action",
        "start_period","start_clock","end_period","end_clock",
        "score_start_for","score_start_against","score_end_for","score_end_against",
        "lineup_pos_mix","lineup_size","players_on_court","team_name"
    ]]
    stints_pm_df = stints_pm_df.merge(
    st_attrs,
    on=["team_no","start_action","end_action"],
    how="left",
    suffixes=("", "_st"),    # <- make the right-hand columns distinct
    validate="many_to_one"
)

    # -------- Lineup vs Lineup matchup segments --------
    lineup_matchups_df = build_lineup_matchups(
        stints_df=stints_df,
        possum_df=possum_df,
        team_nos=team_nos,
        team_map=team_map,
        player_map=player_map,
        pbp_df=pbp_df
    )

    # players + positions table (wide, includes s* and pos normalization)
    players_rows = []
    for tno, t in raw.get("tm", {}).items():
        for pid, p in t.get("pl", {}).items():
            row = {
                "team_no": int(tno),
                "team_name": t.get("name"),
                "team_code": t.get("code"),
                "player_no": int(pid),
            }
            for k in ["name","firstName","familyName","shirtNumber","starter","captain","active"]:
                row[k] = p.get(k)
            pos_guess = coalesce_position_dict(p)
            pos_raw, pos_group = normalize_position_label(pos_guess)
            row["position_raw"] = pos_raw
            row["position_group"] = pos_group
            for k, v in p.items():
                if k.startswith("s"):
                    row[k] = v
            players_rows.append(row)
    players_df = pd.DataFrame(players_rows)

    # Optional: apply overrides if provided
    if OVERRIDE.exists() and not players_df.empty:
        ov = pd.read_csv(OVERRIDE)
        ov.columns = [c.strip() for c in ov.columns]
        if "position" in ov.columns:
            ov["position_raw"], ov["position_group"] = zip(*ov["position"].astype(str).map(normalize_position_label))
        # 1) player_no
        if "player_no" in ov.columns:
            players_df = players_df.merge(
                ov[["player_no","position_raw","position_group"]],
                on="player_no", how="left", suffixes=("","_ov1")
            )
            players_df["position_raw"]   = players_df["position_raw_ov1"].combine_first(players_df["position_raw"])
            players_df["position_group"] = players_df["position_group_ov1"].combine_first(players_df["position_group"])
            players_df.drop(columns=[c for c in players_df.columns if c.endswith("_ov1")], inplace=True)
        # 2) (team_code, shirtNumber)
        if {"team_code","shirtNumber"}.issubset(ov.columns):
            players_df = players_df.merge(
                ov[["team_code","shirtNumber","position_raw","position_group"]],
                on=["team_code","shirtNumber"], how="left", suffixes=("","_ov2")
            )
            players_df["position_raw"]   = players_df["position_raw_ov2"].combine_first(players_df["position_raw"])
            players_df["position_group"] = players_df["position_group_ov2"].combine_first(players_df["position_group"])
            players_df.drop(columns=[c for c in players_df.columns if c.endswith("_ov2")], inplace=True)
        # 3) name
        if "name" in ov.columns:
            players_df = players_df.merge(
                ov[["name","position_raw","position_group"]],
                on="name", how="left", suffixes=("","_ov3")
            )
            players_df["position_raw"]   = players_df["position_raw_ov3"].combine_first(players_df["position_raw"])
            players_df["position_group"] = players_df["position_group_ov3"].combine_first(players_df["position_group"])
            players_df.drop(columns=[c for c in players_df.columns if c.endswith("_ov3")], inplace=True)

        # reflect into player_map (for any downstream use)
        for _, r in players_df[["player_no","position_raw","position_group"]].iterrows():
            if int(r["player_no"]) in player_map:
                player_map[int(r["player_no"])]["playingPosition_raw"] = r["position_raw"]
                player_map[int(r["player_no"])]["position_group"] = r["position_group"]

        # refresh pbp position columns and stints mix (if overrides changed groups)
        pbp_df["player_pos_raw"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("playingPosition_raw") if pd.notna(p) else None)
        pbp_df["player_pos_group"] = pbp_df["pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)
        pbp_df["assist_pos_group"] = pbp_df["assist_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group") if pd.notna(p) else None)
        pbp_df["block_pos_group"]  = pbp_df["block_pno"].map(lambda p: player_map.get(int(p), {}).get("position_group")  if pd.notna(p) else None)
        stints_df["lineup_pos_mix"] = stints_df["players_on_court"].map(lambda lst: lineup_pos_mix(lst, player_map))

    # a small positions-only lookup
    pos_lookup = players_df[[
        "team_no","team_name","team_code","player_no","name","shirtNumber","position_raw","position_group"
    ]].sort_values(["team_no","name"])

    # -----------------------------------------------------------------
    # WRITE
    # -----------------------------------------------------------------
    OUT = Path(".")
    OUT.mkdir(parents=True, exist_ok=True)
    pbp_df.to_csv(OUT/"game_pbp_normalized.csv", index=False)
    possum_df.to_csv(OUT/"game_possessions.csv", index=False)
    team_pm.to_csv(OUT/"game_team_pm.csv", index=False)
    stints_df.to_csv(OUT/"game_stints.csv", index=False)
    stints_pm_df.to_csv(OUT/"game_stints_pm.csv", index=False)
    lineup_matchups_df.to_csv(OUT/"game_lineup_matchups.csv", index=False)
    players_df.to_csv(OUT/"game_players_with_positions.csv", index=False)
    pos_lookup.to_csv(OUT/"game_player_positions_lookup.csv", index=False)

    # Quick console summary
    total_players = len(players_df)
    known_groups = players_df["position_group"].notna() & (players_df["position_group"]!="")
    print(f"[positions] {known_groups.sum()}/{total_players} players have a normalized position group.")
    print(f"[stints] wrote timing & score snapshots.")
    print(f"[matchups] wrote lineup-vs-lineup segments.")

if __name__ == "__main__":
    main()


[positions] 0/24 players have a normalized position group.
[stints] wrote timing & score snapshots.
[matchups] wrote lineup-vs-lineup segments.
